## Hyper-tuning parameters for Logistic Regression, Random Forest, and XGBoost models

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, fbeta_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("blocking_candidates_k40_features_labeled.csv")
df

,src_id,cand_id,cosine_sim,src_text,cand_text,edit_ratio,jaro_winkler,lcs_ratio,token_jaccard,token_cosine,tfidf_word_cosine,tfidf_char_cosine,dmetaphone_match,label
0,3,1,0.192480,Bell Laboratories,"Bell Labs, Lucent Technologies",0.565217,0.686379,0.448276,0.200000,0.353553,0.190326,0.286517,1.0,1
1,22,1,0.544731,"Bell Laboratories, Lucent Technologies","Bell Labs, Lucent Technologies",0.878788,0.894688,0.783784,0.600000,0.750000,0.543631,0.755633,1.0,1
2,100,1,0.169633,Bell Laboratories (India),"Bell Labs, Lucent Technologies",0.500000,0.656472,0.448276,0.166667,0.288675,0.131455,0.228267,1.0,0
3,1,103,0.129490,"Bell Labs, Lucent Technologies",AT&T Labs-Research,0.425532,0.582375,0.344828,0.142857,0.250000,0.127600,0.148219,0.0,0
4,347,1,0.163322,"Data Mining Technologies, Oracle","Bell Labs, Lucent Technologies",0.533333,0.641789,0.516129,0.142857,0.250000,0.105045,0.222851,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62071,9709,9708,0.127536,"Università Roma Tre -- Roma, Italy","Università della Basilicata -- Potenza, Italy",0.647887,0.835407,0.560976,0.285714,0.338062,0.117766,0.143885,1.0,0
62072,9726,9721,0.112279,"Shanghai Jiao Tong University, China","IBM China Research Lab, China",0.412698,0.611905,0.371429,0.125000,0.338062,0.095908,0.120483,0.0,0
62073,9735,9721,0.128880,"Zhejiang University, Hangzhou, China","IBM China Research Lab, China",0.451613,0.605447,0.411765,0.142857,0.377964,0.099372,0.105495,0.0,0
62074,9735,9726,0.068080,"Zhejiang University, Hangzhou, China","Shanghai Jiao Tong University, China",0.666667,0.744412,0.657143,0.285714,0.447214,0.063358,0.166959,0.0,0


In [5]:
df.isnull().sum()

,0
src_id,0
cand_id,0
cosine_sim,0
src_text,0
cand_text,0
edit_ratio,0
jaro_winkler,0
lcs_ratio,0
token_jaccard,0
token_cosine,0


In [6]:
feature_cols = [
    "edit_ratio", "jaro_winkler", "lcs_ratio",
    "token_jaccard", "token_cosine",
    "tfidf_word_cosine", "tfidf_char_cosine",
    "dmetaphone_match",
]

X = df[feature_cols].astype(float).fillna(0.0)
y = df["label"].astype(int)

In [7]:
# As we decided to prioritise recall, we will use fbeta scorer
fbeta = make_scorer(fbeta_score, beta=2)

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Logistic Regression



In [9]:
logreg = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=2000)

param_grid_lr = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l1", "l2"]
}

In [10]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", logreg)
])

grid_lr = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid_lr,
    scoring=fbeta,
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_lr.fit(X, y)
print("Best LR params:", grid_lr.best_params_)
print("Best F2-score:", grid_lr.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best LR params: {'clf__C': 0.01, 'clf__penalty': 'l1'}
Best F2-score: 0.8180581616862614


### Random Forest

In [11]:
rf = RandomForestClassifier(class_weight="balanced", random_state=42)

param_grid_rf = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [10, 20],
    "clf__min_samples_split": [2, 5, 8],
}

In [12]:
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", rf)
])

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring=fbeta,
    cv=cv,
    n_jobs=1,
    verbose=2
)

grid_rf.fit(X, y)
print("Best RF params:", grid_rf.best_params_)
print("Best F2-score:", grid_rf.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=200; total time=  21.9s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=200; total time=  21.7s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=200; total time=  22.8s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=200; total time=  22.8s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=200; total time=  22.6s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=400; total time=  45.8s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=400; total time=  42.4s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=400; total time=  43.8s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=400; total time=  43.5s
[CV] END clf__max_depth=10, clf__min_samples_split=2, clf__n_estimators=400

### XGBoost

In [22]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=y.value_counts()[0] / y.value_counts()[1],
    random_state=42,
    n_jobs=-1
)

param_grid_xgb = {
    "clf__n_estimators": [300, 450, 600],
    "clf__max_depth": [4, 6, 8],
    "clf__learning_rate": [0.01, 0.1, 0.5],
    "clf__subsample": [0.5, 0.9],
    "clf__colsample_bytree": [0.8, 0.9],
}

In [23]:
pipe_xgb = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", xgb)
])

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring=fbeta,
    cv=cv,
    n_jobs=1,
    verbose=2
)

grid_xgb.fit(X, y)
print("Best XGB params:", grid_xgb.best_params_)
print("Best F2-score:", grid_xgb.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.5; total time=   1.5s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.5; total time=   1.5s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.5; total time=   1.4s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.5; total time=   1.5s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.5; total time=   1.4s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estimators=300, clf__subsample=0.9; total time=   1.7s
[CV] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=4, clf__n_estim

### Summarizing results

In [24]:
import pandas as pd

results = pd.DataFrame([
    {"Model": "Logistic Regression", "Best Score (F2)": grid_lr.best_score_, "Params": grid_lr.best_params_},
    {"Model": "Random Forest", "Best Score (F2)": grid_rf.best_score_, "Params": grid_rf.best_params_},
    {"Model": "XGBoost", "Best Score (F2)": grid_xgb.best_score_, "Params": grid_xgb.best_params_},
])

display(results)

,Model,Best Score (F2),Params
0,Logistic Regression,0.818058,"{'clf__C': 0.01, 'clf__penalty': 'l1'}"
1,Random Forest,0.843529,"{'clf__max_depth': 10, 'clf__min_samples_split..."
2,XGBoost,0.853187,"{'clf__colsample_bytree': 0.8, 'clf__learning_..."
